# SB3 Soft Actor-Critic (SAC) — Sequential Pose Correction

This notebook trains a **Soft Actor-Critic (SAC)** agent with **stable-baselines3** on a **sequence observation** (e.g. pose landmarks over time).

Key idea: SB3 SAC is not recurrent by default, but it *can* learn from sequences by using a **custom PyTorch feature extractor** that consumes an observation shaped like `(T, D)`.

Note: if your current Python environment has TensorFlow installed with an incompatible NumPy (common on mixed TF/PyTorch setups), importing SB3 can fail through TensorBoard/TensorFlow. The install cell below pins `numpy<2.1` and `tensorboard<2.19` to avoid that.

**You should adapt the environment** section to your real pose-correction dynamics + reward. The SB3 wiring (policy, extractor, save/load, eval) is the main deliverable.


In [1]:
# If running in a clean env, install deps.
# You can skip this if your venv already has these.

%pip -q install "numpy<2.1" "tensorboard<2.19" "stable-baselines3[extra]" gymnasium torch matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch as th
import torch.nn as nn

from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import VecMonitor

th.manual_seed(0)
np.random.seed(0)

## Environment: sequential pose-correction (template)

This is a **minimal template** environment that matches a common pose-correction framing:

- **Observation**: last `T` frames of pose features, shape `(T, D)`.
- **Action**: continuous vector `a` that proposes a correction direction.
- **Goal**: minimize a hidden target displacement vector (synthetic here).

Replace `step()` with your real reward and dynamics.


In [3]:
@dataclass
class SeqEnvConfig:
    T: int = 30          # sequence length
    D: int = 36          # features per timestep (e.g. 12 landmarks * 3 coords)
    action_dim: int = 36 # typically match D if you output per-feature deltas
    max_steps: int = 50
    target_scale: float = 0.05
    action_scale: float = 0.05


class SequentialPoseCorrectionEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 30}

    def __init__(self, cfg: SeqEnvConfig):
        super().__init__()
        self.cfg = cfg

        self.observation_space = gym.spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(self.cfg.T, self.cfg.D),
            dtype=np.float32,
        )

        self.action_space = gym.spaces.Box(
            low=-1.0,
            high=1.0,
            shape=(self.cfg.action_dim,),
            dtype=np.float32,
        )

        self._step = 0
        self._target = None
        self._seq = None

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self._step = 0

        # Synthetic example: random "true displacement" we want to match.
        self._target = self.np_random.normal(0.0, self.cfg.target_scale, size=(self.cfg.action_dim,)).astype(np.float32)

        # Observation sequence: noisy pose-like features.
        self._seq = self.np_random.normal(0.0, 1.0, size=(self.cfg.T, self.cfg.D)).astype(np.float32)

        info: Dict[str, Any] = {"target": self._target.copy()}
        return self._seq.copy(), info

    def step(self, action: np.ndarray):
        self._step += 1

        action = np.asarray(action, dtype=np.float32)
        action = np.clip(action, -1.0, 1.0)
        proposed = action * self.cfg.action_scale

        # Reward: negative distance to target (maximize reward => match target).
        err = proposed - self._target
        reward = -float(np.mean(err * err))

        terminated = False
        truncated = self._step >= self.cfg.max_steps

        # Update observation with tiny drift + noise to keep it sequential.
        new_frame = (0.98 * self._seq[-1] + self.np_random.normal(0.0, 0.02, size=(self.cfg.D,))).astype(np.float32)
        self._seq = np.concatenate([self._seq[1:], new_frame[None, :]], axis=0)

        info: Dict[str, Any] = {
            "mse": float(np.mean(err * err)),
            "target": self._target.copy(),
            "proposed": proposed.copy(),
        }
        return self._seq.copy(), reward, terminated, truncated, info


## Sequence feature extractor (PyTorch)

SAC’s default policy is an MLP, so we plug in a `features_extractor_class` that consumes `(T, D)` and produces a fixed-size embedding.

This implementation uses small temporal Conv1D blocks + mean pooling.


In [4]:
class TemporalConvExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 256):
        # observation_space shape: (T, D)
        super().__init__(observation_space, features_dim)
        T, D = observation_space.shape
        self.T = int(T)
        self.D = int(D)

        # Conv1d expects (N, C, L). Treat D as channels, T as length.
        self.net = nn.Sequential(
            nn.Conv1d(self.D, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.proj = nn.Sequential(
            nn.Linear(64, features_dim),
            nn.ReLU(),
        )

    def forward(self, observations: th.Tensor) -> th.Tensor:
        # observations: (N, T, D) from gymnasium Box
        x = observations
        if x.ndim != 3:
            raise ValueError(f"Expected (N,T,D) observations, got {tuple(x.shape)}")

        x = x.transpose(1, 2)  # (N, D, T)
        x = self.net(x)        # (N, 64, T)
        x = x.mean(dim=-1)     # (N, 64)
        return self.proj(x)    # (N, features_dim)

## Train SAC (stable-baselines3)

We wrap the env with `Monitor` and vectorize it. Then we train SAC with our extractor.


In [5]:
cfg = SeqEnvConfig(T=30, D=36, action_dim=36, max_steps=50)

def make_env():
    return Monitor(SequentialPoseCorrectionEnv(cfg))

venv = make_vec_env(make_env, n_envs=8, seed=0)
venv = VecMonitor(venv)

policy_kwargs = dict(
    features_extractor_class=TemporalConvExtractor,
    features_extractor_kwargs=dict(features_dim=256),
    net_arch=dict(pi=[256, 256], qf=[256, 256]),
)

model = SAC(
    policy="MlpPolicy",
    env=venv,
    policy_kwargs=policy_kwargs,
    learning_rate=3e-4,
    buffer_size=200_000,
    batch_size=512,
    tau=0.02,
    gamma=0.98,
    train_freq=(1, "step"),
    gradient_steps=1,
    ent_coef="auto",
    verbose=1,
)

model

/opt/anaconda3/envs/true_form/lib/python3.12/site-packages/stable_baselines3/common/vec_env/vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Using cpu device


In [6]:
total_timesteps = 50_000
model.learn(total_timesteps=total_timesteps, progress_bar=True)

/opt/anaconda3/envs/true_form/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -0.159   |
| time/              |          |
|    episodes        | 4        |
|    fps             | 14       |
|    time_elapsed    | 28       |
|    total_timesteps | 400      |
| train/             |          |
|    actor_loss      | -43.8    |
|    critic_loss     | 14.8     |
|    ent_coef        | 0.989    |
|    ent_coef_loss   | -0.655   |
|    learning_rate   | 0.0003   |
|    n_updates       | 37       |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -0.159   |
| time/              |          |
|    episodes        | 8        |
|    fps             | 14       |
|    time_elapsed    | 28       |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_me

KeyboardInterrupt: 

## Evaluate + visualize


In [ ]:
eval_env = SequentialPoseCorrectionEnv(cfg)

def run_episode(deterministic: bool = True):
    obs, info = eval_env.reset(seed=123)
    target = info["target"]

    rewards = []
    mses = []

    for _ in range(cfg.max_steps):
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, r, terminated, truncated, info = eval_env.step(action)
        rewards.append(float(r))
        mses.append(float(info["mse"]))
        if terminated or truncated:
            break

    return np.asarray(rewards), np.asarray(mses), target

rewards, mses, target = run_episode(deterministic=True)
print("episode len:", len(rewards))
print("final mse:", float(mses[-1]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(rewards)
ax[0].set_title("Reward per step")
ax[0].set_xlabel("step")
ax[0].set_ylabel("reward")

ax[1].plot(mses)
ax[1].set_title("MSE(proposed vs target) per step")
ax[1].set_xlabel("step")
ax[1].set_ylabel("mse")

plt.tight_layout()
plt.show()

## Save / load

SB3 saves the policy and replay buffer separately (buffer is optional). For most workflows, saving just the policy is enough.


In [ ]:
out_dir = Path("/Users/yasinetawfeek/Developer/TrueForm/AI/pose_correction/models_sb3")
out_dir.mkdir(parents=True, exist_ok=True)

model_path = out_dir / "sac_sequential_policy"
model.save(str(model_path))

loaded = SAC.load(str(model_path), env=venv)
loaded

## Adapting this to your displacement NPZ

Your repo already has a displacement dataset and metadata used by Keras notebooks.

To train SAC on it, you typically:

- Load `training_data_displacement.npz` and `training_data_displacement_metadata.json`
- Construct each observation as `(T, pose_feats_per_step)` (or `(T, D)` after adding class features)
- Define a per-step action semantics (e.g. correction vector for the last frame)
- Define reward (e.g. negative distance between proposed correction and ground-truth displacement)

Then your environment becomes a supervised-RL hybrid: the agent learns a stochastic policy that maps sequences to continuous corrections.
